# Exercise 10.2 — Impulse Response & Convolution

**목표:**
- [Open AIR Library](http://www.openairlib.net)에서 Impulse Response(IR) 다운로드
- 짧은 녹음(dry recording)을 준비
- 두 가지 방법으로 공간 음향 시뮬레이션:
  1. **Method 1** — 시간 영역 직접 합성곱 (Convolution)
  2. **Method 2** — 주파수 영역 DFT 곱셈 (FFT-based Filtering)

---

### 배경 이론

공간의 음향 특성은 **Impulse Response (IR)** $h[n]$으로 표현된다.

어떤 소리 $x[n]$이 해당 공간에서 들릴 때의 출력 $y[n]$:

$$y[n] = x[n] * h[n] = \sum_{k} x[k] \cdot h[n-k]$$

**합성곱 정리(Convolution Theorem)** 에 의해 주파수 영역에서는 단순 곱셈:

$$Y[f] = X[f] \cdot H[f]$$

따라서 두 방법의 결과는 이론적으로 동일해야 한다.

---
## Step 0. 파일 준비 안내

### Impulse Response 다운로드
1. http://www.openairlib.net 접속
2. **Impulse Response Data** 섹션 탐색
3. 관심 있는 공간 선택 (예: 대성당, 콘서트홀, 동굴 등)
4. WAV 형식으로 다운로드
5. 이 노트북과 같은 폴더에 저장

### Dry Recording 준비
- IR과 **동일한 sample rate**의 WAV 파일 준비
- 짧은 음성, 악기, 효과음 등 권장 (5초 이하)
- [freesound.org](http://freesound.org) 등에서 다운로드 가능

> **아래 파일명을 실제 파일명으로 변경하세요:**
> - `ir_file = 'your_impulse_response.wav'`
> - `dry_file = 'your_dry_recording.wav'`

---
## Step 1. Import 및 파일 로드

In [ ]:
import thinkdsp
import numpy as np
import matplotlib.pyplot as plt
from numpy.fft import rfft, irfft, rfftfreq

%matplotlib inline

# ======================================================
#  파일명을 실제 다운로드한 파일명으로 변경하세요
ir_file  = 'your_impulse_response.wav'   # Open AIR에서 다운로드한 IR
dry_file = 'your_dry_recording.wav'      # 시뮬레이션할 dry 녹음
# ======================================================

# 파일 로드
ir_wave  = thinkdsp.read_wave(ir_file)
dry_wave = thinkdsp.read_wave(dry_file)

print('=== Impulse Response ===')
print(f'  Framerate : {ir_wave.framerate} Hz')
print(f'  Duration  : {ir_wave.duration:.3f} sec')
print(f'  Samples   : {len(ir_wave.ys)}')
print()
print('=== Dry Recording ===')
print(f'  Framerate : {dry_wave.framerate} Hz')
print(f'  Duration  : {dry_wave.duration:.3f} sec')
print(f'  Samples   : {len(dry_wave.ys)}')
print()

# Sample rate 일치 확인
if ir_wave.framerate == dry_wave.framerate:
    print(f'✅ Sample rate 일치: {ir_wave.framerate} Hz')
else:
    print(f'❌ Sample rate 불일치! IR={ir_wave.framerate}Hz, Dry={dry_wave.framerate}Hz')
    print('   → 두 파일의 sample rate를 동일하게 맞춰야 합니다.')

---
## Step 2. 원본 신호 시각화 및 청취

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 6))

# Impulse Response 파형
ir_wave.plot()
plt.sca(axes[0])
ir_wave.plot()
axes[0].set_title('Impulse Response (IR) — 공간의 음향 특성')
axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('Amplitude')
axes[0].grid(True, alpha=0.3)

# Dry Recording 파형
plt.sca(axes[1])
dry_wave.plot()
axes[1].set_title('Dry Recording — 원본 소리 (잔향 없음)')
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Amplitude')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# IR 청취
print('▶ Impulse Response 재생:')
ir_wave.make_audio()

In [ ]:
# Dry recording 청취
print('▶ Dry Recording 재생 (원본):')
dry_wave.make_audio()

---
## Step 3. Method 1 — 시간 영역 합성곱 (Convolution)

$$y[n] = x[n] * h[n] = \sum_{k=0}^{K-1} x[k] \cdot h[n-k]$$

- `np.convolve()` 또는 `thinkdsp`의 합성곱 사용
- 출력 길이 = `len(x) + len(h) - 1`

In [ ]:
dry_ys = dry_wave.ys.astype(float)
ir_ys  = ir_wave.ys.astype(float)

# Stereo IR이면 mono로 변환 (채널 평균)
if ir_ys.ndim == 2:
    ir_ys = ir_ys.mean(axis=1)
    print('IR: Stereo → Mono 변환 완료')

# 정규화 (클리핑 방지)
dry_ys = dry_ys / np.max(np.abs(dry_ys))
ir_ys  = ir_ys  / np.max(np.abs(ir_ys))

# 시간 영역 합성곱
print('합성곱 계산 중... (시간이 걸릴 수 있습니다)')
conv_ys = np.convolve(dry_ys, ir_ys)
print(f'합성곱 완료!')
print(f'  입력 길이  : {len(dry_ys)} samples')
print(f'  IR 길이    : {len(ir_ys)} samples')
print(f'  출력 길이  : {len(conv_ys)} samples  (= {len(dry_ys)} + {len(ir_ys)} - 1)')

# 출력 정규화
conv_ys = conv_ys / np.max(np.abs(conv_ys))

# thinkdsp Wave 객체 생성
conv_wave = thinkdsp.Wave(ys=conv_ys, framerate=dry_wave.framerate)
print(f'  출력 길이  : {conv_wave.duration:.3f} sec')

In [ ]:
# Method 1 결과 파형 시각화
conv_wave.plot()
plt.title('Method 1: 시간 영역 합성곱 결과 (Convolution)')
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 청취
print('▶ Method 1 결과 재생 (Convolution):')
conv_wave.make_audio()

---
## Step 4. Method 2 — 주파수 영역 DFT 곱셈 (FFT-based Filtering)

**합성곱 정리 (Convolution Theorem):**

$$Y[f] = X[f] \cdot H[f]$$

$$y[n] = \mathcal{F}^{-1}\{X[f] \cdot H[f]\}$$

- 두 신호를 같은 길이로 zero-padding 후 FFT
- 주파수 영역에서 point-wise 곱셈
- IFFT로 다시 시간 영역으로 변환

In [ ]:
# Zero-padding: 두 신호를 선형 합성곱 출력 길이로 패딩
N = len(dry_ys) + len(ir_ys) - 1   # 선형 합성곱 출력 길이

# FFT (zero-padded to length N)
X = rfft(dry_ys, n=N)   # Dry recording의 DFT
H = rfft(ir_ys,  n=N)   # IR의 DFT (= Transfer Function)

print(f'Zero-padding 길이 N = {N}')
print(f'DFT 크기: X={X.shape}, H={H.shape}')

# 주파수 영역 곱셈 Y[f] = X[f] * H[f]
Y = X * H

# IFFT로 시간 영역 복원
fft_ys = irfft(Y, n=N)[:N]

# 정규화
fft_ys = fft_ys / np.max(np.abs(fft_ys))

# thinkdsp Wave 객체 생성
fft_wave = thinkdsp.Wave(ys=fft_ys, framerate=dry_wave.framerate)

print(f'FFT 곱셈 완료!')
print(f'  출력 길이: {len(fft_ys)} samples = {fft_wave.duration:.3f} sec')

In [ ]:
# Method 2 결과 파형 시각화
fft_wave.plot()
plt.title('Method 2: DFT 곱셈 결과 (FFT-based Filtering)')
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 청취
print('▶ Method 2 결과 재생 (FFT Filtering):')
fft_wave.make_audio()

---
## Step 5. 두 방법 비교

In [ ]:
# 파형 비교 (처음 2초만 시각화)
compare_len = min(int(2.0 * dry_wave.framerate), len(conv_ys), len(fft_ys))
t = np.arange(compare_len) / dry_wave.framerate

fig, axes = plt.subplots(3, 1, figsize=(13, 9))

axes[0].plot(t, dry_ys[:compare_len], color='steelblue', linewidth=0.8)
axes[0].set_title('원본 Dry Recording')
axes[0].set_ylabel('Amplitude')
axes[0].grid(True, alpha=0.3)

axes[1].plot(t, conv_ys[:compare_len], color='#e74c3c', linewidth=0.8)
axes[1].set_title('Method 1: 시간 영역 합성곱 (Convolution)')
axes[1].set_ylabel('Amplitude')
axes[1].grid(True, alpha=0.3)

axes[2].plot(t, fft_ys[:compare_len], color='#2ecc71', linewidth=0.8)
axes[2].set_title('Method 2: DFT 곱셈 (FFT Filtering)')
axes[2].set_xlabel('Time (s)')
axes[2].set_ylabel('Amplitude')
axes[2].grid(True, alpha=0.3)

plt.suptitle('세 신호 파형 비교 (처음 2초)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Method 1 vs Method 2 수치 차이 계산
min_len = min(len(conv_ys), len(fft_ys))
diff    = conv_ys[:min_len] - fft_ys[:min_len]

print('=== Method 1 vs Method 2 수치 비교 ===')
print(f'  Max absolute difference : {np.max(np.abs(diff)):.6e}')
print(f'  Mean absolute difference: {np.mean(np.abs(diff)):.6e}')
print(f'  RMS difference          : {np.sqrt(np.mean(diff**2)):.6e}')
print()
if np.max(np.abs(diff)) < 1e-5:
    print('✅ 두 방법의 결과가 수치적으로 거의 동일합니다 (합성곱 정리 검증!)')
else:
    print('⚠️  미세한 수치 차이 존재 (부동소수점 오차 범위 내)')

In [ ]:
# 차이(diff) 파형 시각화
t_diff = np.arange(min_len) / dry_wave.framerate

plt.figure(figsize=(13, 3))
plt.plot(t_diff[:compare_len], diff[:compare_len],
         color='orange', linewidth=0.8)
plt.title('Method 1 − Method 2 차이 (수치 오차)')
plt.xlabel('Time (s)')
plt.ylabel('Difference')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Step 6. 스펙트럼 비교

In [ ]:
# 각 신호의 스펙트럼 계산
freqs_dry  = rfftfreq(len(dry_ys),  1 / dry_wave.framerate)
freqs_conv = rfftfreq(len(conv_ys), 1 / dry_wave.framerate)
freqs_fft  = rfftfreq(len(fft_ys),  1 / dry_wave.framerate)

amp_dry  = np.abs(rfft(dry_ys))
amp_conv = np.abs(rfft(conv_ys))
amp_fft  = np.abs(rfft(fft_ys))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(freqs_dry,  amp_dry,  color='steelblue', linewidth=0.8)
axes[0].set_title('Dry Recording Spectrum')
axes[0].set_xlim(0, 8000)
axes[0].set_xlabel('Frequency (Hz)')
axes[0].set_ylabel('Amplitude')
axes[0].grid(True, alpha=0.3)

axes[1].plot(freqs_conv, amp_conv, color='#e74c3c', linewidth=0.8)
axes[1].set_title('Method 1 Spectrum (Convolution)')
axes[1].set_xlim(0, 8000)
axes[1].set_xlabel('Frequency (Hz)')
axes[1].grid(True, alpha=0.3)

axes[2].plot(freqs_fft,  amp_fft,  color='#2ecc71', linewidth=0.8)
axes[2].set_title('Method 2 Spectrum (FFT Filtering)')
axes[2].set_xlim(0, 8000)
axes[2].set_xlabel('Frequency (Hz)')
axes[2].grid(True, alpha=0.3)

plt.suptitle('Spectrum 비교', fontsize=13)
plt.tight_layout()
plt.show()

---
## Step 7. IR의 Transfer Function 시각화

In [ ]:
# H[f] = DFT of Impulse Response = Transfer Function of the space
freqs_ir = rfftfreq(len(ir_ys), 1 / ir_wave.framerate)
H_full   = rfft(ir_ys)
H_amp    = np.abs(H_full)
H_phase  = np.angle(H_full)

fig, axes = plt.subplots(2, 1, figsize=(13, 7))

# 진폭 스펙트럼 (Magnitude Response)
axes[0].plot(freqs_ir, H_amp, color='steelblue', linewidth=0.8)
axes[0].set_title('Transfer Function H[f] — Magnitude Response')
axes[0].set_xlabel('Frequency (Hz)')
axes[0].set_ylabel('|H[f]|')
axes[0].set_xlim(0, ir_wave.framerate // 2)
axes[0].grid(True, alpha=0.3)

# 위상 스펙트럼 (Phase Response)
axes[1].plot(freqs_ir, H_phase, color='#e67e22', linewidth=0.5, alpha=0.7)
axes[1].set_title('Transfer Function H[f] — Phase Response')
axes[1].set_xlabel('Frequency (Hz)')
axes[1].set_ylabel('Phase (radians)')
axes[1].set_xlim(0, ir_wave.framerate // 2)
axes[1].grid(True, alpha=0.3)

plt.suptitle('공간의 Transfer Function (Impulse Response의 DFT)', fontsize=13)
plt.tight_layout()
plt.show()

---
## Step 8. 최종 결과 비교 청취

In [ ]:
print('▶ 1. 원본 Dry Recording:')
dry_wave.make_audio()

In [ ]:
print('▶ 2. Method 1 — 시간 영역 합성곱 결과:')
conv_wave.make_audio()

In [ ]:
print('▶ 3. Method 2 — DFT 곱셈 결과:')
fft_wave.make_audio()

---
## Step 9. 결론 및 분석

### Q1. 두 방법의 결과 비교
- Method 1 (합성곱)과 Method 2 (DFT 곱셈)의 결과가 동일한가?
- 수치 오차(diff)의 크기는 어느 정도인가?

*(분석 내용 작성)*

---

### Q2. 음향 효과 분석
- 원본(dry)과 처리된 소리의 차이는?
- 어떤 공간의 IR을 사용했으며, 해당 공간의 음향 특성이 소리에 어떻게 반영되었는가?

*(분석 내용 작성)*

---

### Q3. 합성곱 정리 (Convolution Theorem) 검증
- 시간 영역의 합성곱 = 주파수 영역의 곱셈 임이 확인되었는가?
- 이론적으로 왜 두 방법이 동일한 결과를 내는가?

*(분석 내용 작성)*

---

### Q4. 두 방법의 계산 효율성
- 시간 영역 합성곱의 계산 복잡도: $O(N \cdot M)$
- FFT 기반 방법의 계산 복잡도: $O(N \log N)$
- IR이 길어질수록 어떤 방법이 더 효율적인가?

*(분석 내용 작성)*